<a href="https://colab.research.google.com/github/sukhadaghewari/tropicalcyclone-ri-bench/blob/main/notebooks/02_make_prompt_files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build RI prompts for the LLM

This notebook turns storms from the data CSV into LLM prompts for rapid-intensification (RI) prediction, as two separate sets of storms:

| Set | Storms | Seasons | Used for |
|---|---|---|---|
| Development | 20 with RI + 20 without | 2016 | writing and debugging the prompt, choosing a decision threshold |
| Test | 50 with RI + 50 without | 2017–2025 | the final evaluation, run once |

1. Load the data and drop storms with only one observation.
2. Sample each set at random with a fixed seed. The sets cannot share a storm, and every development storm comes before every test storm.
3. For every observation of a sampled storm, build one prompt whose history covers that storm's rows from its first observation up to the current one, so the history grows by one row per prompt. Keep the true `ri_tplus24` label for each prompt.
4. Write each set to its own compressed JSON Lines file in `model_inputs/ri_prediction/`: one record per observation, holding its metadata, true label, image paths and prompt.

The prompts never show the storm's position (`lat`, `lon`, `abs_lat`), ID, name, season or date: with the position, LLMs can recognise a storm and recall what it did.

## Setup

The next cell works out the repo folder and makes the code in `src/` importable, so the notebook runs the same way locally and in Colab. In Colab it first clones this repo and moves into it.

In [ ]:
# Setup: in Colab, clone the repo and move into it. Then put src/ on the import path.
import os, subprocess, sys
from pathlib import Path

REPO = "tropicalcyclone-ri-bench"
if "google.colab" in sys.modules:
    try:
        os.getcwd()
    except FileNotFoundError:              # the folder we were in was deleted (e.g. an old clone): start again
        os.chdir("/content")
if "google.colab" in sys.modules and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", f"https://github.com/sukhadaghewari/{REPO}.git"], check=True)
    os.chdir(REPO)
if "google.colab" in sys.modules:
    # A clone from an earlier run survives "Restart session": bring it up to date with GitHub.
    if subprocess.run(["git", "pull", "--ff-only"]).returncode != 0:
        print("WARNING: could not update the repo with git pull; using the files as they are.")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # repo root, locally or in Colab
sys.path.insert(0, str(ROOT / "src"))

The prompt logic lives in `src/prompts.py`:

- `FEATURES`: the columns shown to the model, in the order they appear in each history record. Past intensity is given only as changes (`usa_wind_delta_tminus*`), not as the raw past wind values. `storm_speed` is left out because it is computed from the track positions on both sides of t, so it carries information from after t.
- `TARGET`: the true label, 1 if `usa_wind` rises by 30 kt or more over the next 24 hours.
- `HIDDEN_LOCATION_FEATURES`: `lat`, `lon` and `abs_lat`. They are never shown to the model, because with the track's position LLMs can recognise the storm; `prompts.py` stops with an error if one of them is added to `FEATURES`.
- `IMAGE_CROPS`: the three satellite image crops at time t (`core`, `mid`, `wide`).
- `prepare_storm`, `build_prompt`, `prompt_for_row`, `iter_prompts`, `write_cases` and `parse_response`, described below where they are used.

`prompts.py` loads the prompt template and feature dictionary from `prompt/`, and checks that the dictionary describes exactly `FEATURES`, in the same order.

In [ ]:
import pandas as pd

from prompts import TARGET, prepare_storm, prompt_for_row, write_cases, parse_response

## Settings and data

`SEED` fixes the random sampling, so the same storms are chosen on every run. Storms with only one observation are dropped first: they have no history to reason over.

In [ ]:
SEED = 42
DEV_SEASONS = [2016]
TEST_SEASONS = list(range(2017, 2026))     # 2017-2025
N_DEV_PER_GROUP = 20                       # 20 RI + 20 no-RI storms
N_TEST_PER_GROUP = 50                      # 50 RI + 50 no-RI storms

# keep_default_na=False: otherwise pandas reads the North Atlantic basin code "NA" as a missing value.
df_all = pd.read_csv(ROOT / "data" / "ibtracs_era5_sat_2016_2025.csv", keep_default_na=False, na_values=[""])
print("all data:", df_all.shape[0], "rows,", df_all["sid"].nunique(), "storms")

rows_per_storm = df_all.groupby("sid")["sid"].transform("size")
df_multi = df_all[rows_per_storm > 1].copy()
print("storms with 1 observation removed:", df_all["sid"].nunique() - df_multi["sid"].nunique())
print("remaining:", df_multi.shape[0], "rows,", df_multi["sid"].nunique(), "storms")

## One row per storm

`storms` has one row per storm: its season, and whether it has at least one RI observation (`has_ri`). Sampling works on this table. The counts show how many storms each set can be drawn from.

In [ ]:
storms = df_multi.groupby("sid").agg(season=("season", "first"), has_ri=(TARGET, "max"))

for name, seasons in [("development", DEV_SEASONS), ("test", TEST_SEASONS)]:
    pool = storms[storms["season"].isin(seasons)]
    print(f"{name} pool ({min(seasons)}-{max(seasons)}): {int(pool['has_ri'].sum())} storms with RI, "
          f"{int((pool['has_ri'] == 0).sum())} without")

## Sample the two sets

`sample_storms` draws `n` storms with RI and `n` without from the given seasons. The development and test seasons don't overlap, so the sets can't share a storm. The checks below confirm it, and that development comes first, both by season and by actual date (a Southern Hemisphere season starts in July of the previous year, so the season label alone isn't enough).

In [ ]:
def sample_storms(storms, seasons, n_per_group, seed):
    """Randomly pick n_per_group storms with RI and n_per_group without, from the given seasons."""
    pool = storms[storms["season"].isin(seasons)]
    ri_sids = pool.index[pool["has_ri"] == 1].to_series().sample(n_per_group, random_state=seed).tolist()
    no_ri_sids = pool.index[pool["has_ri"] == 0].to_series().sample(n_per_group, random_state=seed).tolist()
    return ri_sids, no_ri_sids


dev_ri, dev_no_ri = sample_storms(storms, DEV_SEASONS, N_DEV_PER_GROUP, SEED)
test_ri, test_no_ri = sample_storms(storms, TEST_SEASONS, N_TEST_PER_GROUP, SEED)

dev_sids, test_sids = dev_ri + dev_no_ri, test_ri + test_no_ri
assert not set(dev_sids) & set(test_sids), "a storm is in both sets"
assert storms.loc[dev_sids, "season"].max() < storms.loc[test_sids, "season"].min(), "development is not before test"
# Southern Hemisphere seasons start in July of the previous year, so also check the actual dates.
obs_time = pd.to_datetime(df_multi["iso_time"])
assert obs_time[df_multi["sid"].isin(dev_sids)].max() < obs_time[df_multi["sid"].isin(test_sids)].min(), \
    "a test observation is dated before the last development observation"
print("development:", len(dev_sids), "storms | test:", len(test_sids), "storms | no overlap")

## The evaluation dataframes

`make_eval_df` stacks every observation of the chosen storms, sorted by storm and then by time. `prepare_storm` adds `obs_index` (0-based position of the row in its storm) and `hours_since_start`; `storm_group` records whether the storm has RI (`ri`) or not (`no_ri`).

Each row of `df_dev` and `df_test_set` becomes one prompt, and its `ri_tplus24` is that prompt's true label.

In [ ]:
def make_eval_df(df, ri_sids, no_ri_sids):
    """All observations of the given storms, one storm after another in time order."""
    df_eval = pd.concat([prepare_storm(df, sid) for sid in ri_sids + no_ri_sids], ignore_index=True)
    df_eval["storm_group"] = df_eval["sid"].isin(ri_sids).map({True: "ri", False: "no_ri"})
    assert df_eval["usa_wind"].notna().all(), "current usa_wind is missing for some rows; RI is undefined there"
    assert df_eval[TARGET].notna().all(), "missing RI labels"
    return df_eval


df_dev = make_eval_df(df_multi, dev_ri, dev_no_ri)
df_test_set = make_eval_df(df_multi, test_ri, test_no_ri)

for name, d in [("development", df_dev), ("test", df_test_set)]:
    print(f"\n{name}: {len(d)} rows (= prompts) from {d['sid'].nunique()} storms")
    print(d.groupby("storm_group").agg(storms=("sid", "nunique"), rows=("sid", "size"), ri_rows=(TARGET, "sum")))

## Prompt construction

The functions are in `src/prompts.py`. In short:

- `build_prompt(history)` fills the two placeholders in `prompt/ri_prediction.md`: the feature dictionary and the storm history. The history is written as a JSON array with one record per line; missing values become `null` and floats are rounded to 6 significant digits.
- `prompt_for_row(df_eval, i)` returns the prompt and true label for row `i` of a dataframe like `df_dev`. The history is that storm's rows from `obs_index` 0 up to and including row `i`.
- `iter_prompts(df_eval)` yields `(row_index, prompt, label)` for every row, building each prompt only when it is reached.
- `write_cases(df_eval, path, split)` writes one record per observation: metadata, true label, image paths and the prompt.
- `parse_response(text)` reads the first JSON object in the model's reply (ignoring any text around it) and checks it follows the output rules: exactly the keys `reason` and `ri_probability`, in either order, with `reason` a string and the probability a number from 0 to 1.

The model returns only a probability, not a yes/no call. Any decision threshold is chosen afterwards on the development set, never on the test set.

## Sanity check

Build the first two prompts of one development storm with RI and confirm that the history grows by one row and that no location feature appears.

In [ ]:
i0 = df_dev.index[df_dev["sid"] == dev_ri[0]][0]
for i in (i0, i0 + 1):
    prompt, label = prompt_for_row(df_dev, i)
    assert not any(f'"{k}"' in prompt for k in ["lat", "lon", "abs_lat"]), "a location feature is in the prompt"
    print(f"row {i} | sid {df_dev.loc[i, 'sid']} | obs_index {df_dev.loc[i, 'obs_index']} | {TARGET} = {label}")
    print(prompt[prompt.index("<storm_history>"):], "\n")

## Write the case files

Each set goes to its own compressed JSON Lines file (`.jsonl.gz`) in `model_inputs/ri_prediction/`, one record per observation ("case"):

- **Key:** `case_id` = storm ID + 3-digit observation number, e.g. `2016240N13224_007`. Use it to join model results back to cases.
- **Metadata:** `split`, `sid`, `name`, `season`, `basin`, `iso_time`, `obs_index`, `hours_since_start`, `storm_group`. Bookkeeping only; none of it appears in a prompt.
- **Label:** `ri_tplus24`.
- **Images:** `image_t_core`, `image_t_mid`, `image_t_wide`: paths to the satellite image at time t in three crop sizes, relative to `model_inputs/` (e.g. `images/mid/2016240N13224_007_mid.png`). Always filled in; whether an image was actually made is recorded in `model_inputs/images/manifest.csv` by `04_download_satellite_images.ipynb`.
- **Prompt:** `prompt`, the full text sent to the model.

The files are compressed because the prompts are long and repetitive (about 20 times smaller). Reload with `pd.read_json(path, lines=True, convert_dates=False, dtype={"sid": str, "iso_time": str, "basin": str, "case_id": str})`; pandas unzips `.gz` automatically. The extra arguments keep the North Atlantic basin code `NA` and `iso_time` as text (otherwise pandas reads `NA` as missing and turns `iso_time` into a date).

In [ ]:
DEV_PATH = ROOT / "model_inputs" / "ri_prediction" / "dev" / "dev_2016_20ri_20nori_text_only.jsonl.gz"
TEST_PATH = ROOT / "model_inputs" / "ri_prediction" / "test" / "test_2017-2025_50ri_50nori_text_only.jsonl.gz"

for df_eval, path, split in [(df_dev, DEV_PATH, "dev"), (df_test_set, TEST_PATH, "test")]:
    write_cases(df_eval, path, split)
    print(f"saved {len(df_eval)} {split} cases to {path.relative_to(ROOT)} ({path.stat().st_size / 1e6:.1f} MB compressed)")

### RI prompts with satellite pictures

One more version of the RI cases, to send together with each case's combined picture (made by `06_combine_satellite_images.ipynb`):

- **`text_image`:** the RI prompt (`prompt/ri_prediction_text_image.md`), with a section that explains the picture: its layout, the grey scale, magenta and artefact lines.

Each record is the RI case record with this prompt, `prompt_variant`, and `image`, the picture's path relative to `model_inputs/` (`images/combined/<case_id>_combined.png`). The answer format is the RI format, so `parse_response` reads it. Notebooks 07a and 07b attach the picture, after the prompt text, whenever a case has an `image` field.

In [ ]:
from prompts import write_image_cases

for df_eval, split, name in [(df_dev, "dev", "dev_2016_20ri_20nori"), (df_test_set, "test", "test_2017-2025_50ri_50nori")]:
    for variant in ["text_image"]:
        path = ROOT / "model_inputs" / "ri_prediction" / split / f"{name}_{variant}.jsonl.gz"
        write_image_cases(df_eval, path, split, variant)
        print(f"saved {len(df_eval)} {split} cases to {path.relative_to(ROOT)} ({path.stat().st_size / 1e6:.1f} MB compressed)")

## Record the split

Two small CSV files in `model_inputs/splits/` record exactly which storms are in each set, so the split can be checked, cited and reused:

- **`storms.csv`:** one row per storm: set, storm ID, name, season, basin, group (`ri` or `no_ri`), number of cases, number of RI-labelled cases, and its first and last observation time.
- **`summary.csv`:** one row per set: storms (with and without RI), cases (with and without an RI label), RI share, date range, and storms per basin.

If the seed, the data or the filtering ever change, these files change too, so a `git diff` shows it straight away.

In [ ]:
def storm_table(df_eval, split):
    """One row per storm in df_eval."""
    return (df_eval.groupby("sid", sort=False)
            .agg(name=("name", "first"), season=("season", "first"), basin=("basin", "first"),
                 storm_group=("storm_group", "first"), n_cases=("sid", "size"), n_ri_cases=(TARGET, "sum"),
                 first_time=("iso_time", "min"), last_time=("iso_time", "max"))
            .reset_index()
            .assign(split=split))


split_storms = pd.concat([storm_table(df_dev, "dev"), storm_table(df_test_set, "test")], ignore_index=True)
split_storms = split_storms[["split", "sid", "name", "season", "basin", "storm_group", "n_cases", "n_ri_cases", "first_time", "last_time"]]
split_storms = split_storms.sort_values(["split", "storm_group", "sid"]).reset_index(drop=True)

summary = split_storms.groupby("split").agg(
    storms=("sid", "size"),
    ri_storms=("storm_group", lambda g: int((g == "ri").sum())),
    no_ri_storms=("storm_group", lambda g: int((g == "no_ri").sum())),
    cases=("n_cases", "sum"),
    ri_cases=("n_ri_cases", "sum"),
    first_time=("first_time", "min"),
    last_time=("last_time", "max"),
)
summary["no_ri_cases"] = summary["cases"] - summary["ri_cases"]
summary["ri_share"] = (summary["ri_cases"] / summary["cases"]).round(3)
basins = split_storms.pivot_table(index="split", columns="basin", values="sid", aggfunc="count", fill_value=0).add_prefix("storms_")
summary = summary.join(basins).reset_index()
summary = summary[["split", "storms", "ri_storms", "no_ri_storms", "cases", "ri_cases", "no_ri_cases", "ri_share",
                   "first_time", "last_time"] + list(basins.columns)]

# Must agree with the case files written above.
assert summary.set_index("split").loc["dev", "cases"] == len(df_dev) and summary.set_index("split").loc["test", "cases"] == len(df_test_set)

(ROOT / "model_inputs" / "splits").mkdir(parents=True, exist_ok=True)
split_storms.to_csv(ROOT / "model_inputs" / "splits" / "storms.csv", index=False)
summary.to_csv(ROOT / "model_inputs" / "splits" / "summary.csv", index=False)
print(f"saved model_inputs/splits/storms.csv ({len(split_storms)} storms) and model_inputs/splits/summary.csv")
summary

## Using it in an LLM loop

A sketch of the evaluation loop, reading a saved case file. `call_llm` stands in for your model client. Run it on the development file while developing; run it on the test file once, after the prompt and threshold are frozen. Results are keyed by `case_id`.

In [ ]:
# cases = pd.read_json(DEV_PATH, lines=True, convert_dates=False,
#                       dtype={"sid": str, "iso_time": str, "basin": str, "case_id": str})
# results = []
# for _, case in cases.iterrows():
#     raw = call_llm(case["prompt"])
#     try:
#         out = parse_response(raw)
#     except ValueError as e:
#         out = {"parse_error": str(e)}
#     results.append({"case_id": case["case_id"], **out})
# df_results = cases[["case_id", TARGET]].merge(pd.DataFrame(results), on="case_id")